# 04. PD Validation and Scorecard

This final notebook reports probability of default (PD), discrimination metrics, a confusion matrix, and an illustrative 300–850 scorecard.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../data/loan_data_2007_2014.csv')
data = pd.read_csv(DATA_PATH, low_memory=False)
bad_statuses = {'Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)'}
data['good_bad'] = (~data['loan_status'].isin(bad_statuses)).astype(int)
data['annual_inc'] = data['annual_inc'].fillna(data['annual_inc'].median())
data['dti'] = data['dti'].fillna(data['dti'].median())
model_data = pd.get_dummies(data[['grade', 'verification_status', 'term', 'int_rate', 'annual_inc', 'dti', 'good_bad']], columns=['grade', 'verification_status', 'term'], dtype=int).dropna()
X = model_data.drop(columns=['good_bad', 'grade_G'], errors='ignore')
y = model_data['good_bad']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000, solver='liblinear').fit(X_train, y_train)
probability_good = model.predict_proba(X_test)[:, 1]
# PD = 1 - P(good)
PD = 1 - probability_good
pd_predictions = pd.DataFrame({'actual_good_bad': y_test, 'P(good)': probability_good, 'PD': PD}, index=y_test.index)
pd_predictions.head()

## Validation: AUC, Gini, and confusion matrix

AUC and Gini measure rank ordering; the confusion matrix depends on the selected threshold.

In [ ]:
auc = roc_auc_score(y_test, probability_good)
gini = 2 * auc - 1
predicted_good = (probability_good >= 0.5).astype(int)
confusion = confusion_matrix(y_test, predicted_good)
pd.DataFrame({'metric': ['AUC', 'Gini'], 'value': [auc, gini]}), pd.DataFrame(confusion, index=['actual bad', 'actual good'], columns=['predicted bad', 'predicted good'])

## 300–850 scorecard

This linear rescaling is an illustrative interpretation layer. It is not a production scorecard calibration or lending policy.

In [ ]:
min_score, max_score = 300, 850
logit_good = np.log(np.clip(probability_good, 1e-6, 1 - 1e-6) / np.clip(PD, 1e-6, 1 - 1e-6))
low, high = np.quantile(logit_good, [0.01, 0.99])
score = (min_score + (logit_good - low) * (max_score - min_score) / (high - low)).clip(min_score, max_score).round().astype(int)
scorecard = pd.DataFrame({'score': score, 'PD': PD, 'P(good)': probability_good}, index=y_test.index)
scorecard.head(10)

In [ ]:
scorecard['score'].plot.hist(bins=40, title='Illustrative 300–850 score distribution')
scorecard[['score', 'PD']].corr()

## Historical and educational limitations

These results use historical Lending Club observations and a simplified target definition. They do not establish present-day performance, fairness, calibration, stability, approval rules, or regulatory suitability. Validate and govern any real PD or scorecard model separately.